# DeepSeek mHC（Manifold-Constrained Hyper-Connections）

源码导航：[core/residual/mhc.py](../../../core/residual/mhc.py) 中的 `ManifoldHyperConnections`、`PreNormBlockWithMHC`、`sinkhorn_knopp`。

mHC 将残差流从单向量扩展为 **n 路并行流** $x \in \mathbb{R}^{n \times C}$，每层学习读 ($\mathcal{H}^{pre}$)、写 ($\mathcal{H}^{post}$)、混合 ($\mathcal{H}^{res}$) 映射。$\mathcal{H}^{res}$ 经 Sinkhorn-Knopp 投影到双随机矩阵，谱范数 $\le 1$，避免 Hyper-Connections 的信号爆炸。

当 $n=1$ 时退化为标准残差连接。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.residual.mhc import (
    ManifoldHyperConnections, expand_to_streams, collapse_streams, sinkhorn_knopp,
)
from core.norm.rmsnorm import RMSNorm

### 2. Sinkhorn 双随机投影

In [ ]:
logits = torch.randn(4, 4)
M = sinkhorn_knopp(logits, num_iters=20)
print('row sums:', M.sum(-1))
print('col sums:', M.sum(-2))

### 3. mHC 包裹子层

In [ ]:
torch.manual_seed(0)
B, T, C, n = 2, 8, 64, 4
x = torch.randn(B, T, C)
x_streams = expand_to_streams(x, n)

mhc = ManifoldHyperConnections(C, n_streams=n)
sublayer = nn.Linear(C, C, bias=False)
norm = RMSNorm(C)

out = mhc(x_streams, sublayer, norm=norm)
assert out.shape == (B, T, n, C)
print('streams out:', tuple(out.shape))
print('collapsed:', tuple(collapse_streams(out).shape))

---

## 延伸阅读

- DeepSeek mHC (2025). [arXiv:2512.24880](https://arxiv.org/abs/2512.24880)
- Hyper-Connections (2024). [arXiv:2409.19606](https://arxiv.org/abs/2409.19606)
- 注意：mHC 与 MLA 不同——MLA 是注意力内的 KV 压缩，mHC 是残差拓扑。